# Sprint 10 - Leaf Segmentation Quick Test

**Hypothesis:** PlantDoc field photos have cluttered backgrounds (dirt, other plants,
hands, shadows) that confuse the disease classifier. If we strip the background,
both datasets look more similar (just a leaf), and the model's field score should rise.

**This sprint is a cheap test — no retraining.** We segment the 230 PlantDoc test
images, then run our existing best model on them to see if field F1 jumps.

**Approach:**
- PlantVillage: already has clean backgrounds (lab photos) — no segmentation needed
- PlantDoc: use `rembg` with `isnet-general-use` model to strip backgrounds
- Run existing checkpoints on segmented test images
- Compare F1 before vs after

**Success criterion:** field F1 on segmented PD test >= 0.50 (up from 0.42)

**If this works:** retrain on segmented data in Sprint 11
**If this fails:** fall back to Plan A (sweep PlantDoc repeat values on EfficientNet-B0)

In [ ]:
import os
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    print(gpu.stdout.strip().splitlines()[0] if gpu.stdout.strip() else gpu.stderr.strip() or "No GPU detected (CPU only)")
except Exception as exc:
    print("GPU check skipped:", exc)

## Step 1 - Mount Drive + clone repo + install deps

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/io-PEAK/folium.git"
REPO_DIR = Path("/content/folium")
DATA_DIR = Path("/content/drive/MyDrive/folium/data")
LOCAL_RAW_DIR = Path("/content/folium_raw")
LOCAL_DATA_DIR = Path("/content/folium_data")
CHECKPOINT_DIR = Path("/content/drive/MyDrive/folium/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/folium/results")

if not (REPO_DIR / "ml").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

for d in (DATA_DIR, LOCAL_RAW_DIR, LOCAL_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("DATA_DIR (archives):", DATA_DIR)
print("LOCAL_DATA_DIR:", LOCAL_DATA_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

def run(cmd, cwd, label, stream=False):
    env = dict(os.environ, PYTHONPATH=str(REPO_DIR))
    if stream:
        proc = subprocess.Popen(
            cmd, cwd=str(cwd), env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        lines = []
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
        proc.wait()
        combined = "".join(lines)
        if proc.returncode != 0:
            print(f"\n[{label}] failed (returncode {proc.returncode})")
        assert proc.returncode == 0, label
        class _Result:
            pass
        r = _Result()
        r.stdout = combined
        r.stderr = ""
        r.returncode = 0
        return r
    result = subprocess.run(cmd, cwd=str(cwd), env=env, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"[{label}] failed (returncode {result.returncode})")
        print("stdout tail:\n", result.stdout[-2000:])
        print("stderr tail:\n", result.stderr[-2000:])
    assert result.returncode == 0, label
    return result

In [ ]:
%pip install -q torch torchvision albumentations matplotlib pandas tqdm opencv-python-headless scikit-learn
%pip install -q "numpy<2" --force-reinstall
%pip install -q "scipy<1.11" --force-reinstall
%pip install -q rembg[cpu]

## Step 2 - Clean old Sprint 9 rows from ablation CSV

In [ ]:
import pandas as pd

csv_path = RESULTS_DIR / "ablation_results.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    old = df["variant"].str.startswith("s9_") | df["variant"].str.startswith("s10_")
    n_old = old.sum()
    if n_old > 0:
        df = df[~old].reset_index(drop=True)
        df.to_csv(csv_path, index=False)
        print(f"Removed {n_old} old s9/s10 rows from {csv_path}")
    else:
        print("No old s9/s10 rows found.")
else:
    print("No ablation CSV yet.")

## Step 3 - Hydrate raw from Drive, then organize splits locally

In [ ]:
import sys

sys.path.insert(0, str(REPO_DIR))
from scripts.download_datasets import PLANTVILLAGE_EXPECTED, PLANTDOC_EXPECTED, hydrate_dataset

for name, expected in (("plantvillage", PLANTVILLAGE_EXPECTED), ("plantdoc", PLANTDOC_EXPECTED)):
    try:
        hydrate_dataset(LOCAL_RAW_DIR, DATA_DIR, name, expected)
    except RuntimeError as exc:
        print("HYDRATE FAILED:", exc)
        raise

result = run([
    sys.executable,
    str(REPO_DIR / "scripts" / "organize_datasets.py"),
    "--raw-dir", str(LOCAL_RAW_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--seed", "42",
    "--val-fraction", "0.15",
    "--test-fraction", "0.15",
], cwd=str(REPO_DIR), label="organize_datasets.py failed")
print("Data ready at", LOCAL_DATA_DIR)

## Step 4 - List available checkpoints

We need the `mixed` checkpoint (v13: fine-tune-then-mix, single-head trained on PV+PD combined).
Also list the dual-head domain checkpoint for comparison.

In [ ]:
ckpts = sorted(CHECKPOINT_DIR.glob("best_*.pt"))
print(f"Found {len(ckpts)} checkpoints:")
for c in ckpts:
    size_mb = c.stat().st_size / 1024 / 1024
    print(f"  {c.name} ({size_mb:.0f}MB)")

## Step 5 - Segment PlantDoc test images

Run `rembg` with `isnet-general-use` on every PlantDoc test image.
Saves to `/tmp/seg_plantdoc_test/<class>/image.jpg` (same folder structure).

In [ ]:
from PIL import Image
from rembg import remove, new_session
from tqdm import tqdm

src_dir = LOCAL_DATA_DIR / "plantdoc" / "test"
dst_dir = Path("/tmp/seg_plantdoc_test")

if dst_dir.exists():
    import shutil
    shutil.rmtree(dst_dir)

session = new_session("isnet-general-use")

# Collect all images first
all_images = []
for class_dir in sorted(src_dir.iterdir()):
    if not class_dir.is_dir():
        continue
    for img_path in sorted(class_dir.glob("*")):
        if img_path.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}:
            all_images.append((class_dir.name, img_path))

print(f"Segmenting {len(all_images)} images...")
count = 0
errors = 0
for class_name, img_path in tqdm(all_images, desc="Segmenting"):
    dst_class = dst_dir / class_name
    dst_class.mkdir(parents=True, exist_ok=True)
    try:
        img = Image.open(img_path).convert("RGB")
        result = remove(img, session=session)
        # remove() returns RGBA — convert to RGB before saving as JPG
        if result.mode == "RGBA":
            bg = Image.new("RGB", result.size, (0, 0, 0))
            bg.paste(result, mask=result.split()[3])
            result = bg
        elif result.mode != "RGB":
            result = result.convert("RGB")
        result.save(dst_class / img_path.name)
        count += 1
    except Exception as e:
        errors += 1
        print(f"  ERROR on {img_path.name}: {e}")

print(f"Done: {count} segmented, {errors} errors")

## Step 6 - Visual spot-check

Render 20 segmented images side-by-side with originals.
Check that the segmenter is removing background, not cutting into leaves.

In [ ]:
import matplotlib.pyplot as plt

src_images = sorted([p for p in src_dir.rglob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])[:20]

n = len(src_images)
cols = 5
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols * 2, figsize=(24, 4 * rows))
if rows == 1:
    axes = [axes]

for i, src_img in enumerate(src_images):
    r, c = divmod(i, cols)
    orig = Image.open(src_img).convert("RGB")
    seg_path = dst_dir / src_img.parent.name / src_img.name
    seg = Image.open(seg_path).convert("RGB") if seg_path.exists() else orig

    axes[r][c * 2].imshow(orig)
    axes[r][c * 2].set_title("original", fontsize=8)
    axes[r][c * 2].axis("off")

    axes[r][c * 2 + 1].imshow(seg)
    axes[r][c * 2 + 1].set_title("segmented", fontsize=8)
    axes[r][c * 2 + 1].axis("off")

# hide unused axes
for i in range(n, rows * cols):
    r, c = divmod(i, cols)
    axes[r][c * 2].axis("off")
    axes[r][c * 2 + 1].axis("off")

plt.tight_layout()
plt.savefig(RESULTS_DIR / "s10_spot_check.png", dpi=120)
print("Saved", RESULTS_DIR / "s10_spot_check.png")
plt.show()

## Step 7 - Evaluate on segmented PlantDoc test

Run existing checkpoints on the segmented test images.
We evaluate:
1. The `mixed` checkpoint (v13, single-head, trained on PV+PD) — our best all-rounder
2. The domain-routed dual-head checkpoint (Sprint 9) — for comparison

We create a temporary data directory with the segmented images + class_map.json,
then point `ml/evaluate.py` at it.

In [ ]:
import json
import shutil

# Create temporary data directory with segmented PD test
seg_data_dir = Path("/tmp/seg_data")
if seg_data_dir.exists():
    shutil.rmtree(seg_data_dir)
seg_data_dir.mkdir()

# Symlink plantdoc/test to segmented images
(seg_data_dir / "plantdoc").mkdir()
(seg_data_dir / "plantdoc" / "test").symlink_to(dst_dir.resolve())

# Copy class_map.json (needed for --map-to-pv)
shutil.copy(LOCAL_DATA_DIR / "class_map.json", seg_data_dir / "class_map.json")

# Also need plantvillage test dir (evaluate may reference it)
pv_test = LOCAL_DATA_DIR / "plantvillage" / "test"
if pv_test.exists():
    (seg_data_dir / "plantvillage").mkdir(exist_ok=True)
    (seg_data_dir / "plantvillage" / "test").symlink_to(pv_test.resolve())

print("Segmented data dir:", seg_data_dir)
if dst_dir.exists():
    print("PD test classes:", sorted([d.name for d in (seg_data_dir / "plantdoc" / "test").iterdir()]))
else:
    print("WARNING: Segmented images not found. Run Step 5 first.")

In [ ]:
# Find the mixed checkpoint (v13)
mixed_candidates = list(CHECKPOINT_DIR.glob("best_plantvillage_mixed*.pt"))
if mixed_candidates:
    MIXED_CKPT = mixed_candidates[0]
    print(f"Using mixed checkpoint: {MIXED_CKPT.name}")
else:
    print("WARNING: No mixed checkpoint found. Available checkpoints:")
    for c in sorted(CHECKPOINT_DIR.glob("best_*.pt")):
        print(f"  {c.name}")
    print("\nSet MIXED_CKPT manually below.")
    MIXED_CKPT = None

In [ ]:
# 1) Mixed checkpoint on segmented PD test (single-head, no --dual-head)
if MIXED_CKPT:
    print("=" * 60)
    print("7a) mixed on SEGMENTED PlantDoc test")
    print("=" * 60)
    cmd = [
        sys.executable, "-m", "ml.evaluate",
        "--checkpoint", str(MIXED_CKPT),
        "--data-dir", str(seg_data_dir),
        "--dataset", "plantdoc",
        "--split", "test",
        "--map-to-pv",
        "--variant", "s10_seg_mixed_field",
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
    ]
    result = run(cmd, cwd=str(REPO_DIR), label="eval failed", stream=True)

    # 2) Same checkpoint on original PD test (baseline comparison)
    print("\n" + "=" * 60)
    print("7b) mixed on ORIGINAL PlantDoc test (baseline)")
    print("=" * 60)
    cmd = [
        sys.executable, "-m", "ml.evaluate",
        "--checkpoint", str(MIXED_CKPT),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--dataset", "plantdoc",
        "--split", "test",
        "--map-to-pv",
        "--variant", "s10_orig_mixed_field",
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
    ]
    result = run(cmd, cwd=str(REPO_DIR), label="eval failed", stream=True)

    # 3) Sanity check: same checkpoint on PV test (should not drop)
    print("\n" + "=" * 60)
    print("7c) mixed on PlantVillage test (sanity check, should stay ~0.95)")
    print("=" * 60)
    cmd = [
        sys.executable, "-m", "ml.evaluate",
        "--checkpoint", str(MIXED_CKPT),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--dataset", "plantvillage",
        "--split", "test",
        "--variant", "s10_orig_mixed_lab",
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
    ]
    result = run(cmd, cwd=str(REPO_DIR), label="eval failed", stream=True)

## Step 8 - Verdict

Compare field F1 on segmented vs original PlantDoc test.

**If segmented F1 >= 0.50:** segmentation helps -> proceed to Sprint 11 (retrain on segmented data)

**If segmented F1 < 0.50:** background isn't the main issue -> fall back to Plan A (repeat sweep)

In [ ]:
df = pd.read_csv(str(RESULTS_DIR / "ablation_results.csv")).drop_duplicates()

seg_key = "s10_seg_mixed_field"
orig_key = "s10_orig_mixed_field"
lab_key = "s10_orig_mixed_lab"

seg_row = df[df["variant"] == seg_key]
orig_row = df[df["variant"] == orig_key]
lab_row = df[df["variant"] == lab_key]

print("=== Segmentation effect on field F1 ===")
if len(orig_row) > 0 and len(seg_row) > 0:
    orig_f1 = orig_row["f1"].iloc[0]
    seg_f1 = seg_row["f1"].iloc[0]
    delta = seg_f1 - orig_f1
    print(f"Original PD test:  {orig_f1:.4f}")
    print(f"Segmented PD test: {seg_f1:.4f}")
    print(f"Delta:             {delta:+.4f}")
    if seg_f1 >= 0.50:
        print(f"\nVerdict: PASS ({seg_f1:.4f} >= 0.50) -> segmentation helps, proceed to retraining")
    else:
        print(f"\nVerdict: FAIL ({seg_f1:.4f} < 0.50) -> background not the main bottleneck, try Plan A")
else:
    print("Missing results. Run Step 7 first.")

if len(lab_row) > 0:
    print(f"\nLab sanity check: {lab_row['f1'].iloc[0]:.4f} (should be ~0.95)")
    if lab_row["f1"].iloc[0] < 0.85:
        print("WARNING: lab dropped! Segmenter may be cutting into leaves.")